# 05 - Hybrid Retrievers

Combine vector similarity with fulltext keyword search. Hybrid search helps when queries contain specific technical terms (fault codes, part numbers, exact specs) that keyword matching handles better than semantics alone.

**Prerequisites:** run **03_data_and_embeddings** first (creates both the vector and fulltext indexes).

| Approach | Strength | Weakness |
|----------|----------|----------|
| Vector | Finds related content with different wording | Can miss exact terms |
| Fulltext | Precise keyword matching | Misses semantic matches |
| Hybrid | Both | Slightly more setup |

## Section 1: Configuration

In [ ]:
%pip install neo4j-graphrag mlflow

In [ ]:
# ============================================
# CONFIGURATION
# ============================================

NEO4J_URI = ""
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = ""

if not NEO4J_URI or not NEO4J_PASSWORD:
    print("WARNING: enter your Neo4j credentials above before running!")
else:
    print(f"Configuration ready - {NEO4J_URI}")

In [ ]:
from neo4j_graphrag.retrievers import HybridRetriever, HybridCypherRetriever, VectorRetriever
from neo4j_graphrag.generation import GraphRAG

from data_utils import Neo4jConnection, get_llm, get_embedder

neo4j = Neo4jConnection(NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD).verify()
driver = neo4j.driver

llm = get_llm()
embedder = get_embedder()
VECTOR_INDEX = "maintenanceChunkEmbeddings"
FULLTEXT_INDEX = "maintenanceChunkText"
print(f"LLM: {llm.model_id} | Embedder: {embedder.model_id}")

---

# Part 1: Hybrid Retriever

Query both indexes and merge the results by combined relevance.

In [ ]:
hybrid_retriever = HybridRetriever(
    driver=driver, vector_index_name=VECTOR_INDEX, fulltext_index_name=FULLTEXT_INDEX,
    embedder=embedder, return_properties=["text"],
)

query = "V2500 engine EGT exceedance troubleshooting"
result = hybrid_retriever.search(query_text=query, top_k=5)
print(f'Query: "{query}"  ({len(result.items)} results)\n' + "=" * 70)
for item in result.items:
    print(f"\nscore={item.metadata['score']:.4f}")
    print(f"  {item.content[:200]}...")

In [ ]:
query = "What are the V2500 engine operating limits and what should I do if EGT is exceeded?"

rag = GraphRAG(llm=llm, retriever=hybrid_retriever)
response = rag.search(
    query, retriever_config={"top_k": 5}, return_context=True,
    response_fallback="No relevant maintenance procedures found.",
)
print(f'Query: "{query}"\n' + "=" * 70)
print("\nAnswer:\n" + response.answer)

---

# Part 2: Hybrid Cypher Retriever

`HybridCypherRetriever` adds graph traversal on top of hybrid search: keyword precision, semantic understanding, and structural context in one retriever.

In [ ]:
adjacent_chunks_query = """
WITH node
OPTIONAL MATCH (prev:Chunk)-[:NEXT_CHUNK]->(node)
OPTIONAL MATCH (node)-[:NEXT_CHUNK]->(next:Chunk)
MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)
RETURN doc.documentId AS document_id,
       node.index AS chunk_index,
       COALESCE(prev.text, '') AS previous_context,
       node.text AS main_context,
       COALESCE(next.text, '') AS next_context
"""

hybrid_adjacent_retriever = HybridCypherRetriever(
    driver=driver, vector_index_name=VECTOR_INDEX, fulltext_index_name=FULLTEXT_INDEX,
    embedder=embedder, retrieval_query=adjacent_chunks_query,
)

query = "What is the procedure for hydraulic fluid contamination check?"
rag = GraphRAG(llm=llm, retriever=hybrid_adjacent_retriever)
response = rag.search(
    query, retriever_config={"top_k": 3}, return_context=True,
    response_fallback="No relevant maintenance procedures found.",
)
print(f'Query: "{query}"\n' + "=" * 70)
print("\nAnswer:\n" + response.answer)

---

# Part 3: Vector vs Hybrid comparison

The two manuals cover different engine families (V2500 on the A320, CFM56-7B on the B737). Technical queries that name an engine or a numeric spec are where hybrid search pulls ahead, because the fulltext component catches the exact term across both manuals.

In [ ]:
vector_retriever = VectorRetriever(
    driver=driver, index_name=VECTOR_INDEX, embedder=embedder, return_properties=["text"],
)

comparison_queries = [
    {"label": "Semantic (natural language)", "query": "How do I fix an engine that is running too hot?"},
    {"label": "Technical - A320 (V2500)", "query": "V2500 EGT limit 925 degrees"},
    {"label": "Technical - B737 (CFM56)", "query": "CFM56-7B N1 speed limit"},
]

for entry in comparison_queries:
    print(f"\n{'=' * 70}\n{entry['label']}: \"{entry['query']}\"\n" + "=" * 70)
    v = vector_retriever.search(query_text=entry["query"], top_k=3)
    print(f"  [Vector] score={v.items[0].metadata['score']:.4f}: {v.items[0].content[:110]}...")
    h = hybrid_retriever.search(query_text=entry["query"], top_k=3)
    print(f"  [Hybrid] score={h.items[0].metadata['score']:.4f}: {h.items[0].content[:110]}...")

In [ ]:
query = "What fault codes indicate bearing wear in the V2500 engine?"
print(f'Query: "{query}"\n' + "=" * 70)

print("\n[1] VECTOR RETRIEVER\n" + "-" * 40)
print(GraphRAG(llm=llm, retriever=vector_retriever).search(
    query, retriever_config={"top_k": 3},
    response_fallback="No relevant maintenance procedures found.",
).answer)

print("\n" + "=" * 70 + "\n[2] HYBRID RETRIEVER\n" + "-" * 40)
print(GraphRAG(llm=llm, retriever=hybrid_retriever).search(
    query, retriever_config={"top_k": 3},
    response_fallback="No relevant maintenance procedures found.",
).answer)

## Summary

| Retriever | Search method | Graph traversal | Best for |
|-----------|---------------|-----------------|----------|
| `VectorRetriever` | Semantic | No | Natural language questions |
| `VectorCypherRetriever` | Semantic | Yes | Context-rich semantic search |
| `HybridRetriever` | Semantic + keyword | No | Technical terminology |
| `HybridCypherRetriever` | Semantic + keyword | Yes | Full-featured technical search |

Use hybrid search when queries carry specific codes, part numbers, or engine designations that exact matching handles better than semantics alone.

In [ ]:
neo4j.close()